In [1]:
import re
import json
import random
import pandas as pd
from elasticsearch import Elasticsearch, helpers

### Read the unique values

In [2]:
# import os
# os.listdir("../dependencies")

In [3]:
   
json_path = '../dependencies/unique_values_22_02_24_preprocessed.json'
with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

In [4]:
grade_list = data['GRADE']
random.shuffle(grade_list)
applications_list = data['APPLICATION']
random.shuffle(applications_list)
Brands = data['BRAND']
random.shuffle(Brands)
Polymer = data['POLYMER']
random.shuffle(Polymer)
property_list = data['PROPERTY']
random.shuffle(property_list)
modifiers_list = ['weak', 'lower', 'inferior', 'low', 'very low',
'standard', 'typical', 'good', 'intermediate', 'common', 'fair', 'moderate', 'normal', 'ordinary',
'maximum', 'best', 'superior', 'elevated', 'great', 'exceptional', 'really good', 'outstanding', 'superb', 'high', 'very high', 'very good', 'highest']

random.shuffle(modifiers_list)
fillers_list = data['FILLER']
random.shuffle(fillers_list)
features_list = data['FEATURE']
random.shuffle(features_list)
competitor_grade_names = data['COMPETITOR_GRADE']
random.shuffle(competitor_grade_names)
certifications_list = data['CERTIFICATION']
random.shuffle(certifications_list)
units_list = data['UNIT']
units_list = list(set(units_list))
random.shuffle(units_list)


competitor_grade_names_without_brand = data['COMP_GRADE_WITHOUT_BRAND']
random.shuffle(competitor_grade_names_without_brand)
grade_list_without_brand = data['GRADE_WITHOUT_BRAND']
random.shuffle(grade_list_without_brand)

grade_list_extended = grade_list + grade_list_without_brand
random.shuffle(grade_list_extended)
competitor_grade_names_extended = competitor_grade_names + competitor_grade_names_without_brand
random.shuffle(competitor_grade_names_extended)

In [5]:
len(property_list)

4556

In [6]:
application_sentencess = ['for my','for','for a','to be used for','to use in','which is for my','product for',' ','with following applications','with applications such as','which can be used in','commonly used in','widely adopted in','ideal for']

items = ['grade','product','material','grades','products','materials']

In [7]:
entity_names = {'grade1': "GRADE",
'grade2': "GRADE",
                
'application1': "APPLICATION",
'application2': "APPLICATION",

'proprty1': "PROPERTY",
'proprty2': "PROPERTY",
'proprty3': "PROPERTY",

'modifier1': "MODIFIER",
'modifier2': "MODIFIER",
'modifier3': "MODIFIER",

'brand1': "BRAND",
'brand2': "BRAND",
                
'polymer_type1': "POLYMER",
'polymer_type2': "POLYMER",
'polymer_type3': "POLYMER",

'feature1': "FEATURE",
'feature2': "FEATURE",
'feature3': "FEATURE",
'feature4': "FEATURE",

'filler_type1': 'FILLER',
'filler_percentage1': 'FILLER_PERCENTAGE',
'filler_type2': 'FILLER',
'filler_percentage2': 'FILLER_PERCENTAGE',

'competitor_grade' :'COMPETITOR_GRADE',

'certification1': 'CERTIFICATION',
'certification2': 'CERTIFICATION',
'certification3': 'CERTIFICATION',
       
'ul_property_name': "PROPERTY",               
'ul_property_name_value': "PROPERTY",
'thickness_property_name': "PROPERTY",
                
'thickness_modifier': "MODIFIER",
'ul_property_value': "MODIFIER"
                
               }

In [8]:
def contains_duplicates(my_list):
    for i in range(len(my_list)):
        for j in range(len(my_list)):
            if i != j and my_list[i] in my_list[j]:
                return True
    return False

In [9]:
import re


with open ('Query templates/user_modifiers_templates.txt', 'r') as file:  
    modifiers_templates=[line.strip() for line in file if line.startswith('f"')]

with open ('Query templates/user_query_templates.txt', 'r') as file:  
    user_query_templates=[line.strip() for line in file if line.startswith('f"')]
    user_query_templates = list(set(user_query_templates))

with open ('Query templates/New templates.txt', 'r') as file:  
    user_query_templates_new=[line.strip() for line in file if line.startswith('f"')]
    user_query_templates += list(set(user_query_templates_new))

with open ('Query templates/user_query_templates_with_special_characters.txt', 'r') as file:  
    additional_user_query_templates=[line.strip() for line in file if line.startswith('f"')]
    additional_user_query_templates = list(set(additional_user_query_templates))
    
with open ('Query templates/user_query_templates grades only sample.txt', 'r') as file:  
    user_query_templates_grades=[line.strip() for line in file if line.startswith('f"')]
    user_query_templates_grades = list(set(user_query_templates_grades))

user_query_templates.extend(additional_user_query_templates)
len(user_query_templates)

2275

In [10]:
len(set(user_query_templates))*290

656270

### Queries with all templates, random values

In [11]:
user_queries = []
for user_query_template in user_query_templates:
    #print(user_query_template)
    for i in range(290):
        application_sample = random.sample(applications_list,2)
        property_sample = random.sample(property_list,3)
        
        modifiers_sample_from_template = random.sample(modifiers_templates,2)
        modifiers_sample = []
        for modifier in modifiers_sample_from_template:
            value = random.choice([str(random.randint(-100, 10000)), str(random.randint(-100, 10000)), str(random.randint(-100, 10000)), str(round(random.uniform(0, 1),2)), random.choice(["{:.1e}","{:.1E}"]).format(random.randint(1, 15) ** random.randint(-10, 15))])
            min_value = random.randint(1, 10000)
            max_value = random.randint(min_value, 10000)
            unit = random.choice([""," "])+random.choice(units_list)
            modifiers_sample.append(eval(modifier))
        modifiers_sample+=random.sample(modifiers_list,2)
        random.shuffle(modifiers_sample)

        feature_sample = random.sample(features_list,4)

        certification_sample = random.sample(certifications_list,3)
        polymer_type_sample = random.sample(Polymer,3)

        filler_type_sample = random.sample(fillers_list,2)
        
        brand_sample = random.sample(Brands,2)
        
        grade_sample = random.sample(grade_list,2)
        
        if contains_duplicates(property_sample) or contains_duplicates(feature_sample) \
                or contains_duplicates(polymer_type_sample) \
                or contains_duplicates(filler_type_sample) \
                or contains_duplicates(grade_sample):
                continue
        else:
            variable_values = {'application1': application_sample[0],
            'application2': application_sample[1],

            'proprty1': property_sample[0],
            'proprty2': property_sample[1],
            'proprty3': property_sample[2],

            'modifier1': modifiers_sample[0],
            'modifier2': modifiers_sample[1],
            'modifier3': modifiers_sample[2],

            'brand1': brand_sample[0],
            'brand2': brand_sample[1],

            'polymer_type1':polymer_type_sample[0],
            'polymer_type2':polymer_type_sample[1],
            'polymer_type3':polymer_type_sample[2],

            'feature1': feature_sample[0],
            'feature2': feature_sample[1],
            'feature3': feature_sample[2],
            'feature4': feature_sample[3],

            'filler_type1': filler_type_sample[0],
            'filler_type2': filler_type_sample[1],

            'filler_percentage1': random.choice(["","","","","","","","","","","","","","~","=","= ",">=",">= ",">","> ","lower than ", "less than ","greater than ","higher than "]) + \
                               str(random.choice([random.choice(range(0,100)), round(random.uniform(0,100),0)])) + \
                               random.choice([" %","%"," percentage"," percent","", "",""]),
            'filler_percentage2': random.choice(["","","","","","","","","","","","","","~","=","= ",">=",">= ",">","> ","lower than ", "less than ","greater than ","higher than "]) + \
                               str(random.choice([random.choice(range(0,100)), round(random.uniform(0,100),0)])) + \
                               random.choice([" %","%"," percentage"," percent","", "",""]),

            'application_sentences': random.choice(application_sentencess),
            'item': random.choice(items),
            'competitor_grade' :random.choice(competitor_grade_names),
            
            'grade1': grade_sample[0],
            'grade2': grade_sample[1],

            'certification1': certification_sample[0],
            'certification2': certification_sample[1],
            'certification3': certification_sample[2]}

            user_query = eval(user_query_template)
            user_query = user_query.lower()
            entities = []
            
            # Find all variables in the f-string
            matches = list(re.finditer(r"{(.*?)}", user_query_template))
            # Calculate the new start/end positions
            entities = []
            offset = 0
            for match in matches:
                variable = match.group(1)
                name_pattern = re.compile(r"\['(.+?)'\]")
                name_of_variable = name_pattern.search(variable).group(1)
                value = eval(variable)
                start_position = match.start()-2 + offset
                end_position = start_position + len(str(value))
                if name_of_variable not in ["item","start_sentence","application_sentences"]:
                    entities.append((start_position, end_position, entity_names[name_of_variable]))
                # Update the offset for the next match
                offset += len(str(value)) - (match.end() - match.start())

            if min([entity[0] for entity in entities])!=-1:
                user_queries.append((user_query, {'entities': entities}))
    #         print((user_query, {'entities': entities}))
    #         print('\n')

In [12]:
random.shuffle(user_queries)
len(user_queries)

626677

In [13]:
from sklearn.model_selection import train_test_split

user_queries_train, user_queries_test = train_test_split( user_queries, test_size=0.06, random_state=42)

len(user_queries_train), len(user_queries_test)

(589076, 37601)

In [14]:
with open ('./NER Model 650K - 22-02-24/annotated/user_queries_train.txt', 'w', encoding="utf-8") as file:  
    for line in user_queries_train: 
        #print(line)
        file.write(str(line))
        file.write('\n')

In [15]:
with open ('./NER Model 650K - 22-02-24/annotated/user_queries_test.txt', 'w', encoding="utf-8") as file:  
    for line in user_queries_test: 
        #print(line)
        file.write(str(line))
        file.write('\n')

### additional queries with  grade names and high, medium, low modifiers 

In [11]:
with open ('./NER Model 650K - 22-02-24/annotated/user_queries_train.txt', 'r',encoding='utf-8' ) as file:  
    train_data=[eval(line.strip()) for line in file]

with open ('./NER Model 650K - 22-02-24/annotated/user_queries_test.txt', 'r',encoding='utf-8' ) as file:  
    test_data=[eval(line.strip()) for line in file]

In [25]:
user_queries = []
for user_query_template in user_query_templates:
    # if "proprty" in user_query_template:
    #     n=8
    # if "grade1" in user_query_template:
    #     n=300
    # if "competitor_grade" in user_query_template:
    #     n=5
#     if "certification1" in user_query_template:
#         n=30
    # if "application" in user_query_template:
    #     n=10
        #print(user_query_template)
        for i in range(n):
            application_sample = random.sample(applications_list,2)
            property_sample = random.sample(property_list,3)


            modifiers_sample=random.sample(modifiers_list,3)

            feature_sample = random.sample(features_list,4)

            certification_sample = random.sample(certifications_list,3)
            polymer_type_sample = random.sample(Polymer,3)

            filler_type_sample = random.sample(fillers_list,2)

            brand_sample = random.sample(Brands,2)

            grade_sample = random.sample(grade_list_extended,2)

            if contains_duplicates(property_sample) or contains_duplicates(feature_sample) \
                    or contains_duplicates(polymer_type_sample) \
                    or contains_duplicates(filler_type_sample) \
                    or contains_duplicates(grade_sample):
                    continue
            else:
                variable_values = {'application1': application_sample[0],
                'application2': application_sample[1],

                'proprty1': property_sample[0],
                'proprty2': property_sample[1],
                'proprty3': property_sample[2],

                'modifier1': modifiers_sample[0],
                'modifier2': modifiers_sample[1],
                'modifier3': modifiers_sample[2],

                'brand1': brand_sample[0],
                'brand2': brand_sample[1],

                'polymer_type1':polymer_type_sample[0],
                'polymer_type2':polymer_type_sample[1],
                'polymer_type3':polymer_type_sample[2],

                'feature1': feature_sample[0],
                'feature2': feature_sample[1],
                'feature3': feature_sample[2],
                'feature4': feature_sample[3],

                'filler_type1': filler_type_sample[0],
                'filler_type2': filler_type_sample[1],

                'filler_percentage1': random.choice(["","","","","","","","","","","~","=","= ",">=",">= ",">","> ","lower than ", "less than ","greater than ","higher than "]) + \
                                   str(random.choice([random.choice(range(0,100)), round(random.uniform(0,100),0)])) + \
                                   random.choice([" %","%"," percentage"," percent","", "",""]),
                'filler_percentage2': random.choice(["","","","","","","","","","","~","=","= ",">=",">= ",">","> ","lower than ", "less than ","greater than ","higher than "]) + \
                                   str(random.choice([random.choice(range(0,100)), round(random.uniform(0,100),0)])) + \
                                   random.choice([" %","%"," percentage"," percent","", "",""]),

                'application_sentences': random.choice(application_sentencess),
                'item': random.choice(items),
                'competitor_grade' :random.choice(competitor_grade_names_extended),

                'grade1': grade_sample[0],
                'grade2': grade_sample[1],

                'certification1': certification_sample[0],
                'certification2': certification_sample[1],
                'certification3': certification_sample[2]}

                user_query = eval(user_query_template)
                user_query = user_query.lower()
                entities = []

                # Find all variables in the f-string
                matches = list(re.finditer(r"{(.*?)}", user_query_template))
                # Calculate the new start/end positions
                entities = []
                offset = 0
                for match in matches:
                    variable = match.group(1)
                    name_pattern = re.compile(r"\['(.+?)'\]")
                    name_of_variable = name_pattern.search(variable).group(1)
                    value = eval(variable)
                    start_position = match.start()-2 + offset
                    end_position = start_position + len(str(value))
                    if name_of_variable not in ["item","start_sentence","application_sentences"]:
                        entities.append((start_position, end_position, entity_names[name_of_variable]))
                    # Update the offset for the next match
                    offset += len(str(value)) - (match.end() - match.start())

                if min([entity[0] for entity in entities])!=-1:
                    user_queries.append((user_query, {'entities': entities}))
        #         print((user_query, {'entities': entities}))
        #         print('\n')

In [26]:
from sklearn.model_selection import train_test_split
user_queries_train, user_queries_test = train_test_split( user_queries, test_size=0.25, random_state=42)
print(len(user_queries_train), len(user_queries_test))

train_data+=user_queries_train
test_data+=user_queries_test
len(train_data), len(test_data)

6114 2038


(622272, 48669)

## Application only queries

In [22]:
user_queries = []
for user_query_template in user_query_templates:
    if user_query_template == 'f"{variable_values[\'application1\']}"':
        n=3000
        for i in range(n):
            application_sample = random.sample(applications_list,2)
            property_sample = random.sample(property_list,3)


            modifiers_sample=random.sample(modifiers_list,3)

            feature_sample = random.sample(features_list,4)

            certification_sample = random.sample(certifications_list,3)
            polymer_type_sample = random.sample(Polymer,3)

            filler_type_sample = random.sample(fillers_list,2)

            brand_sample = random.sample(Brands,2)

            grade_sample = random.sample(grade_list_extended,2)

            if contains_duplicates(property_sample) or contains_duplicates(feature_sample) \
                    or contains_duplicates(polymer_type_sample) \
                    or contains_duplicates(filler_type_sample) \
                    or contains_duplicates(grade_sample):
                    continue
            else:
                variable_values = {'application1': application_sample[0],
                'application2': application_sample[1],

                'proprty1': property_sample[0],
                'proprty2': property_sample[1],
                'proprty3': property_sample[2],

                'modifier1': modifiers_sample[0],
                'modifier2': modifiers_sample[1],
                'modifier3': modifiers_sample[2],

                'brand1': brand_sample[0],
                'brand2': brand_sample[1],

                'polymer_type1':polymer_type_sample[0],
                'polymer_type2':polymer_type_sample[1],
                'polymer_type3':polymer_type_sample[2],

                'feature1': feature_sample[0],
                'feature2': feature_sample[1],
                'feature3': feature_sample[2],
                'feature4': feature_sample[3],

                'filler_type1': filler_type_sample[0],
                'filler_type2': filler_type_sample[1],

                'filler_percentage1': random.choice(["","","","","","","","","","","~","=","= ",">=",">= ",">","> ","lower than ", "less than ","greater than ","higher than "]) + \
                                   str(random.choice([random.choice(range(0,100)), round(random.uniform(0,100),0)])) + \
                                   random.choice([" %","%"," percentage"," percent","", "",""]),
                'filler_percentage2': random.choice(["","","","","","","","","","","~","=","= ",">=",">= ",">","> ","lower than ", "less than ","greater than ","higher than "]) + \
                                   str(random.choice([random.choice(range(0,100)), round(random.uniform(0,100),0)])) + \
                                   random.choice([" %","%"," percentage"," percent","", "",""]),

                'application_sentences': random.choice(application_sentencess),
                'item': random.choice(items),
                'competitor_grade' :random.choice(competitor_grade_names_extended),

                'grade1': grade_sample[0],
                'grade2': grade_sample[1],

                'certification1': certification_sample[0],
                'certification2': certification_sample[1],
                'certification3': certification_sample[2]}

                user_query = eval(user_query_template)
                user_query = user_query.lower()
                entities = []

                # Find all variables in the f-string
                matches = list(re.finditer(r"{(.*?)}", user_query_template))
                # Calculate the new start/end positions
                entities = []
                offset = 0
                for match in matches:
                    variable = match.group(1)
                    name_pattern = re.compile(r"\['(.+?)'\]")
                    name_of_variable = name_pattern.search(variable).group(1)
                    value = eval(variable)
                    start_position = match.start()-2 + offset
                    end_position = start_position + len(str(value))
                    if name_of_variable not in ["item","start_sentence","application_sentences"]:
                        entities.append((start_position, end_position, entity_names[name_of_variable]))
                    # Update the offset for the next match
                    offset += len(str(value)) - (match.end() - match.start())

                if min([entity[0] for entity in entities])!=-1:
                    user_queries.append((user_query, {'entities': entities}))
        #         print((user_query, {'entities': entities}))
        #         print('\n')

In [25]:
from sklearn.model_selection import train_test_split
user_queries_train, user_queries_test = train_test_split( user_queries, test_size=0.2, random_state=42)
print(len(user_queries_train), len(user_queries_test))

train_data+=user_queries_train
test_data+=user_queries_test
len(train_data), len(test_data)

2273 569


(664268, 62485)

In [26]:

with open ('./NER Model 650K - 22-02-24/annotated/user_queries_train.txt', 'w', encoding="utf-8") as file:  
    for line in train_data: 
        #print(line)
        file.write(str(line))
        file.write('\n')


with open ('./NER Model 650K - 22-02-24/annotated/user_queries_test.txt', 'w', encoding="utf-8") as file:  
    for line in test_data: 
        #print(line)
        file.write(str(line))
        file.write('\n')

#### additional queries for filler percentage lower than, higer than, etc

In [27]:
user_queries = []
for user_query_template in user_query_templates:
    if "filler_percentage" in user_query_template:
        #print(user_query_template)
        n=20
        for i in range(n):
            application_sample = random.sample(applications_list,2)
            property_sample = random.sample(property_list,3)

            modifiers_sample_from_template = random.sample(modifiers_templates,2)
            modifiers_sample = []
            for modifier in modifiers_sample_from_template:
                value = random.choice([str(random.randint(-100, 10000)), str(random.randint(-100, 10000)), str(round(random.uniform(0, 1),2)), random.choice(["{:.1e}","{:.1E}"]).format(random.randint(1, 15) ** random.randint(-10, 15))])
                min_value = random.randint(1, 10000)
                max_value = random.randint(min_value, 10000)
                unit = random.choice([""," "])+random.choice(units_list)
                modifiers_sample.append(eval(modifier))
            modifiers_sample+=random.sample(modifiers_list,2)
            random.shuffle(modifiers_sample)

            feature_sample = random.sample(features_list,4)

            certification_sample = random.sample(certifications_list,3)
            polymer_type_sample = random.sample(Polymer,3)

            filler_type_sample = random.sample(fillers_list,2)

            brand_sample = random.sample(Brands,2)

            grade_sample = random.sample(grade_list_extended,2)

            if contains_duplicates(property_sample) or contains_duplicates(feature_sample) \
                    or contains_duplicates(polymer_type_sample) \
                    or contains_duplicates(filler_type_sample) \
                    or contains_duplicates(grade_sample):
                    continue
            else:
                variable_values = {'application1': application_sample[0],
                'application2': application_sample[1],

                'proprty1': property_sample[0],
                'proprty2': property_sample[1],
                'proprty3': property_sample[2],

                'modifier1': modifiers_sample[0],
                'modifier2': modifiers_sample[1],
                'modifier3': modifiers_sample[2],

                'brand1': brand_sample[0],
                'brand2': brand_sample[1],

                'polymer_type1':polymer_type_sample[0],
                'polymer_type2':polymer_type_sample[1],
                'polymer_type3':polymer_type_sample[2],

                'feature1': feature_sample[0],
                'feature2': feature_sample[1],
                'feature3': feature_sample[2],
                'feature4': feature_sample[3],

                'filler_type1': filler_type_sample[0],
                'filler_type2': filler_type_sample[1],

                'filler_percentage1': random.choice(["","","","","","","","","","","","","","~","=","= ",">=",">= ",">","> ","lower than ", "less than ","greater than ","higher than "]) + \
                                   str(random.choice([random.choice(range(0,100)), round(random.uniform(0,100),0)])) + \
                                   random.choice([" %","%"," percentage"," percent","", "",""]),
                'filler_percentage2': random.choice(["","","","","","","","","","","","","","~","=","= ",">=",">= ",">","> ","lower than ", "less than ","greater than ","higher than "]) + \
                                   str(random.choice([random.choice(range(0,100)), round(random.uniform(0,100),0)])) + \
                                   random.choice([" %","%"," percentage"," percent","", "",""]),

                'application_sentences': random.choice(application_sentencess),
                'item': random.choice(items),
                'competitor_grade' :random.choice(competitor_grade_names_extended),

                'grade1': grade_sample[0],
                'grade2': grade_sample[1],

                'certification1': certification_sample[0],
                'certification2': certification_sample[1],
                'certification3': certification_sample[2]}

                user_query = eval(user_query_template)
                user_query = user_query.lower()
                entities = []

                # Find all variables in the f-string
                matches = list(re.finditer(r"{(.*?)}", user_query_template))
                # Calculate the new start/end positions
                entities = []
                offset = 0
                for match in matches:
                    variable = match.group(1)
                    name_pattern = re.compile(r"\['(.+?)'\]")
                    name_of_variable = name_pattern.search(variable).group(1)
                    value = eval(variable)
                    start_position = match.start()-2 + offset
                    end_position = start_position + len(str(value))
                    if name_of_variable not in ["item","start_sentence","application_sentences"]:
                        entities.append((start_position, end_position, entity_names[name_of_variable]))
                    # Update the offset for the next match
                    offset += len(str(value)) - (match.end() - match.start())

                if min([entity[0] for entity in entities])!=-1:
                    user_queries.append((user_query, {'entities': entities}))
        #         print((user_query, {'entities': entities}))
        #         print('\n')

In [28]:
from sklearn.model_selection import train_test_split
user_queries_train, user_queries_test = train_test_split( user_queries, test_size=0.25, random_state=42)
print(len(user_queries_train), len(user_queries_test))

train_data+=user_queries_train
test_data+=user_queries_test
len(train_data), len(test_data)

6234 2079


(628506, 50748)

In [29]:
# i=0
# for query in train_data:
#     if "FILLER_PERCENTAGE" in query and any(item.lower() in query.lower() for item in ["lower than ", "less than ","greater than ","higher than "]):
#         i+=1
# print(i)
        

In [61]:
with open ('./NER Model 650K - 22-02-24/annotated/user_queries_train.txt', 'w', encoding="utf-8") as file:  
    for line in train_data: 
        #print(line)
        file.write(str(line))
        file.write('\n')


with open ('./NER Model 650K - 22-02-24/annotated/user_queries_test.txt', 'w', encoding="utf-8") as file:  
    for line in test_data: 
        #print(line)
        file.write(str(line))
        file.write('\n')

### Additional queries with more features

In [29]:
user_queries = []
for user_query_template in user_query_templates:
    if "feature" in user_query_template:
        #print(user_query_template)
        n=10
        for i in range(n):
            application_sample = random.sample(applications_list,2)
            property_sample = random.sample(property_list,3)

            modifiers_sample_from_template = random.sample(modifiers_templates,2)
            modifiers_sample = []
            for modifier in modifiers_sample_from_template:
                value = random.choice([str(random.randint(-100, 10000)), str(random.randint(-100, 10000)), str(round(random.uniform(0, 1),2)), random.choice(["{:.1e}","{:.1E}"]).format(random.randint(1, 15) ** random.randint(-10, 15))])
                min_value = random.randint(1, 10000)
                max_value = random.randint(min_value, 10000)
                unit = random.choice([""," "])+random.choice(units_list)
                modifiers_sample.append(eval(modifier))
            modifiers_sample+=random.sample(modifiers_list,2)
            random.shuffle(modifiers_sample)
            
            features_list = ["very low warpage", "low emissions","very high flow", "very high heat resistance","good fuel resistance"]
            feature_sample = random.sample(features_list,4)

            certification_sample = random.sample(certifications_list,3)
            polymer_type_sample = random.sample(Polymer,3)

            filler_type_sample = random.sample(fillers_list,2)

            brand_sample = random.sample(Brands,2)

            grade_sample = random.sample(grade_list_extended,2)

            if contains_duplicates(property_sample) or contains_duplicates(feature_sample) \
                    or contains_duplicates(polymer_type_sample) \
                    or contains_duplicates(filler_type_sample) \
                    or contains_duplicates(grade_sample):
                    continue
            else:
                variable_values = {'application1': application_sample[0],
                'application2': application_sample[1],

                'proprty1': property_sample[0],
                'proprty2': property_sample[1],
                'proprty3': property_sample[2],

                'modifier1': modifiers_sample[0],
                'modifier2': modifiers_sample[1],
                'modifier3': modifiers_sample[2],

                'brand1': brand_sample[0],
                'brand2': brand_sample[1],

                'polymer_type1':polymer_type_sample[0],
                'polymer_type2':polymer_type_sample[1],
                'polymer_type3':polymer_type_sample[2],

                'feature1': feature_sample[0],
                'feature2': feature_sample[1],
                'feature3': feature_sample[2],
                'feature4': feature_sample[3],

                'filler_type1': filler_type_sample[0],
                'filler_type2': filler_type_sample[1],

                'filler_percentage1': random.choice(["","","","","","","","","","","","","","~","=","= ",">=",">= ",">","> ","lower than ", "less than ","greater than ","higher than "]) + \
                                   str(random.choice([random.choice(range(0,100)), round(random.uniform(0,100),0)])) + \
                                   random.choice([" %","%"," percentage"," percent","", "",""]),
                'filler_percentage2': random.choice(["","","","","","","","","","","","","","~","=","= ",">=",">= ",">","> ","lower than ", "less than ","greater than ","higher than "]) + \
                                   str(random.choice([random.choice(range(0,100)), round(random.uniform(0,100),0)])) + \
                                   random.choice([" %","%"," percentage"," percent","", "",""]),

                'application_sentences': random.choice(application_sentencess),
                'item': random.choice(items),
                'competitor_grade' :random.choice(competitor_grade_names_extended),

                'grade1': grade_sample[0],
                'grade2': grade_sample[1],

                'certification1': certification_sample[0],
                'certification2': certification_sample[1],
                'certification3': certification_sample[2]}

                user_query = eval(user_query_template)
                user_query = user_query.lower()
                entities = []

                # Find all variables in the f-string
                matches = list(re.finditer(r"{(.*?)}", user_query_template))
                # Calculate the new start/end positions
                entities = []
                offset = 0
                for match in matches:
                    variable = match.group(1)
                    name_pattern = re.compile(r"\['(.+?)'\]")
                    name_of_variable = name_pattern.search(variable).group(1)
                    value = eval(variable)
                    start_position = match.start()-2 + offset
                    end_position = start_position + len(str(value))
                    if name_of_variable not in ["item","start_sentence","application_sentences"]:
                        entities.append((start_position, end_position, entity_names[name_of_variable]))
                    # Update the offset for the next match
                    offset += len(str(value)) - (match.end() - match.start())

                if min([entity[0] for entity in entities])!=-1:
                    user_queries.append((user_query, {'entities': entities}))
        #         print((user_query, {'entities': entities}))
        #         print('\n')

In [30]:
from sklearn.model_selection import train_test_split
user_queries_train, user_queries_test = train_test_split( user_queries, test_size=0.25, random_state=42)
print(len(user_queries_train), len(user_queries_test))

train_data+=user_queries_train
test_data+=user_queries_test
len(train_data), len(test_data)

6609 2204


(635115, 52952)

In [64]:
with open ('./NER Model 650K - 22-02-24/annotated/user_queries_train.txt', 'w', encoding="utf-8") as file:  
    for line in train_data: 
        #print(line)
        file.write(str(line))
        file.write('\n')


with open ('./NER Model 650K - 22-02-24/annotated/user_queries_test.txt', 'w', encoding="utf-8") as file:  
    for line in test_data: 
        #print(line)
        file.write(str(line))
        file.write('\n')

### additional queries with partial grade names

In [15]:
with open ('./NER Model 650K - 22-02-24/annotated/user_queries_train.txt', 'r',encoding='utf-8' ) as file:  
    train_data=[eval(line.strip()) for line in file]

with open ('./NER Model 650K - 22-02-24/annotated/user_queries_test.txt', 'r',encoding='utf-8' ) as file:  
    test_data=[eval(line.strip()) for line in file]
len(train_data), len(test_data)

(650259, 58969)

In [39]:
user_queries = []
for user_query_template in user_query_templates:
    # if "grade1" in user_query_template:
    #     n=300
    # if "competitor_grade" in user_query_template:
    #     n=2
    # if "competitor_grade" in user_query_template and len(user_query_template)<100:
    #     n=50
    # if "application" in user_query_template and "feature" in user_query_template and len(user_query_template)<80:
    #     n=350
    # if "filler_type1" in user_query_template:
    #     n=5
        #print(user_query_template)
        for i in range(n):
            application_sample = random.sample(applications_list,2)
            property_sample = random.sample(property_list,3)

            modifiers_sample=random.sample(modifiers_list,3)

            feature_sample = random.sample(features_list,4)

            certification_sample = random.sample(certifications_list,3)
            polymer_type_sample = random.sample(Polymer,3)

            filler_type_sample = random.sample(fillers_list,2)

            brand_sample = random.sample(Brands,2)

            grade_sample = random.sample(grade_list_without_brand,2)

            if contains_duplicates(property_sample) or contains_duplicates(feature_sample) \
                    or contains_duplicates(polymer_type_sample) \
                    or contains_duplicates(filler_type_sample) \
                    or contains_duplicates(grade_sample):
                    continue
            else:
                variable_values = {'application1': application_sample[0],
                'application2': application_sample[1],

                'proprty1': property_sample[0],
                'proprty2': property_sample[1],
                'proprty3': property_sample[2],

                'modifier1': modifiers_sample[0],
                'modifier2': modifiers_sample[1],
                'modifier3': modifiers_sample[2],

                'brand1': brand_sample[0],
                'brand2': brand_sample[1],

                'polymer_type1':polymer_type_sample[0],
                'polymer_type2':polymer_type_sample[1],
                'polymer_type3':polymer_type_sample[2],

                'feature1': feature_sample[0],
                'feature2': feature_sample[1],
                'feature3': feature_sample[2],
                'feature4': feature_sample[3],

                'filler_type1': filler_type_sample[0],
                'filler_type2': filler_type_sample[1],

                'filler_percentage1': random.choice(["","","","","","","","","","","~","=","= ",">=",">= ",">","> ","lower than ", "less than ","greater than ","higher than "]) + \
                                   str(random.choice([random.choice(range(0,100)), round(random.uniform(0,100),0)])) + \
                                   random.choice([" %","%"," percentage"," percent","", "",""]),
                'filler_percentage2': random.choice(["","","","","","","","","","","~","=","= ",">=",">= ",">","> ","lower than ", "less than ","greater than ","higher than "]) + \
                                   str(random.choice([random.choice(range(0,100)), round(random.uniform(0,100),0)])) + \
                                   random.choice([" %","%"," percentage"," percent","", "",""]),

                'application_sentences': random.choice(application_sentencess),
                'item': random.choice(items),
                'competitor_grade' :random.choice(competitor_grade_names_without_brand),

                'grade1': grade_sample[0],
                'grade2': grade_sample[1],

                'certification1': certification_sample[0],
                'certification2': certification_sample[1],
                'certification3': certification_sample[2]}

                user_query = eval(user_query_template)
                user_query = user_query.lower()
                entities = []

                # Find all variables in the f-string
                matches = list(re.finditer(r"{(.*?)}", user_query_template))
                # Calculate the new start/end positions
                entities = []
                offset = 0
                for match in matches:
                    variable = match.group(1)
                    name_pattern = re.compile(r"\['(.+?)'\]")
                    name_of_variable = name_pattern.search(variable).group(1)
                    value = eval(variable)
                    start_position = match.start()-2 + offset
                    end_position = start_position + len(str(value))
                    if name_of_variable not in ["item","start_sentence","application_sentences"]:
                        entities.append((start_position, end_position, entity_names[name_of_variable]))
                    # Update the offset for the next match
                    offset += len(str(value)) - (match.end() - match.start())

                if min([entity[0] for entity in entities])!=-1:
                    user_queries.append((user_query, {'entities': entities}))
        #         print((user_query, {'entities': entities}))
        #         print('\n')

In [40]:
from sklearn.model_selection import train_test_split
user_queries_train, user_queries_test = train_test_split( user_queries, test_size=0.25, random_state=42)
print(len(user_queries_train), len(user_queries_test))

train_data+=user_queries_train
test_data+=user_queries_test
len(train_data), len(test_data)

3318 1106


(651353, 58366)

In [75]:
with open ('./NER Model 650K - 22-02-24/annotated/user_queries_train.txt', 'w', encoding="utf-8") as file:  
    for line in train_data: 
        #print(line)
        file.write(str(line))
        file.write('\n')


with open ('./NER Model 650K - 22-02-24/annotated/user_queries_test.txt', 'w', encoding="utf-8") as file:  
    for line in test_data: 
        #print(line)
        file.write(str(line))
        file.write('\n')

### additional queries with property modifier mapping done properly

In [41]:
def get_es():
    ELASTIC_PASSWORD = "<ELASTICSEARCH_PASSWORD_DEV>"
    es = Elasticsearch(
        "https://elast-gst-nprd-ussc-01.es.privatelink.southcentralus.azure.elastic-cloud.com:9243",
        basic_auth=("elastic", ELASTIC_PASSWORD))
    print(es.info())
    return es

ES = get_es()
# fetch all ul properties
match_all_query = {
    "query": {
    "match_all": {}
    }
}

ul_resp = ES.search(index='filter_ul', body=match_all_query, size=100)
ul_cert = [hit['_source'] for hit in ul_resp['hits']['hits']]


ul_numerical = []
ul_categorical = []
for item in ul_cert:
    if item['Max_Value']:
        print(item['UL_PROPERTY'],item['Min_Value'],item['Max_Value'],item['UNIT_OF_MEAS_SI'])
#         print("\n")
        ul_numerical.append({x: item[x] for x in item if x not in {"SUB_PROPERTIES"}})
        if item['UL_PROPERTY']=='Minimum Thickness (mm)':
            for sub_item in item['SUB_PROPERTIES']:
                try:
                    print(sub_item['UL_SUB_PROPERTY'],sub_item['Min_Value'],sub_item['Max_Value'],sub_item['UNIT_OF_MEAS_SI'])
#                     print("\n")
                    ul_numerical.append(sub_item)
                except:
                    print(sub_item['UL_SUB_PROPERTY'],sub_item['Categorical_Values'])
#                     print("\n")
                    ul_categorical.append(sub_item)
    else:
        print(item['UL_PROPERTY'],item['Categorical_Values'])
#         print("\n")
        ul_categorical.append(item)
    
    
for item in ul_numerical:
    try: item['UL_PROPERTY']=item['UL_PROPERTY'].replace("(°C)","").replace("(mm)","").strip()
    except: item['UL_SUB_PROPERTY']=item['UL_SUB_PROPERTY'].replace("(°C)","").replace("(mm)","").strip()
        
ul_numerical_additional = []
for item in ul_numerical:
    try: 
        if '(' in item['UL_PROPERTY']:
            item_copy = item.copy()
            new_prop_name = re.findall(r'\((.*?)\)', item_copy['UL_PROPERTY'])[0]
            item_copy['UL_PROPERTY'] = new_prop_name
#             print(item)
#             print(item_copy)
            ul_numerical_additional.append(item_copy)
        
            item_copy = item.copy()
            new_prop_name = item_copy['UL_PROPERTY'].split(" (")[0]
            item_copy['UL_PROPERTY'] = new_prop_name
#             print(item_copy)
            ul_numerical_additional.append(item_copy)
            
    except: 
        if '(' in item['UL_SUB_PROPERTY']:
            item_copy = item.copy()
            new_prop_name = re.findall(r'\((.*?)\)', item_copy['UL_SUB_PROPERTY'])[0]
            item_copy['UL_SUB_PROPERTY'] = new_prop_name
#             print(item)
#             print(item_copy)
            ul_numerical_additional.append(item_copy)
        
            item_copy = item.copy()
            new_prop_name = item_copy['UL_SUB_PROPERTY'].split(" (")[0]
            item_copy['UL_SUB_PROPERTY'] = new_prop_name
#             print(item_copy)
            ul_numerical_additional.append(item_copy)
            
ul_categorical_additional = []

for item in ul_categorical:
    if 'PLC' in item['Categorical_Values'][0]:
            item['Categorical_Values'] = [value.split(' (')[0] for value in item['Categorical_Values']]
    try: 
        if '(' in item['UL_PROPERTY']:
            item_copy = item.copy()
            new_prop_name = re.findall(r'\((.*?)\)', item_copy['UL_PROPERTY'])[0]
            item_copy['UL_PROPERTY'] = new_prop_name
            ul_categorical_additional.append(item_copy)
        
            item_copy = item.copy()
            new_prop_name = item_copy['UL_PROPERTY'].split(" (")[0]
            item_copy['UL_PROPERTY'] = new_prop_name
            ul_categorical_additional.append(item_copy)
            
    except: 
        if '(' in item['UL_SUB_PROPERTY']:
            item_copy = item.copy()
            new_prop_name = re.findall(r'\((.*?)\)', item_copy['UL_SUB_PROPERTY'])[0]
            item_copy['UL_SUB_PROPERTY'] = new_prop_name
            ul_categorical_additional.append(item_copy)
        
            item_copy = item.copy()
            new_prop_name = item_copy['UL_SUB_PROPERTY'].split(" (")[0]
            item_copy['UL_SUB_PROPERTY'] = new_prop_name
            ul_categorical_additional.append(item_copy)

for item in ["High Voltage Arc Tracking Rate", "HVTR"]:
    ul_numerical_additional.append({'UL_PROPERTY': item,
    'Min_Value': 0,
    'Max_Value': 1000,
    'UNIT_OF_MEAS_SI': 'mm/min'})
    
for item in ["Comparative Tracking Index", "CTI"]:
    ul_numerical_additional.append({'UL_PROPERTY': item,
    'Min_Value': 0,
    'Max_Value': 1000,
    'UNIT_OF_MEAS_SI': 'V'})
    
for item in ["Hot Wire Ignition", "HWI", "High Current Arc Ignition", "HAI", "Arc Resistance"]:
    ul_numerical_additional.append({'UL_PROPERTY': item,
    'Min_Value': 0,
    'Max_Value': 1000,
    'UNIT_OF_MEAS_SI': random.choice(['seconds','sec','s'])})
    
ul_categorical = ul_categorical+ul_categorical_additional
ul_numerical = ul_numerical+ul_numerical_additional
# fetch all properties
match_all_query = {
    "query": {
    "match_all": {}
    }
}

prop_resp = ES.search(index='filter_property', body=match_all_query, size=1000)
prop_cert = [hit['_source'] for hit in prop_resp['hits']['hits']]

porp_categorical = []
properties_considered = []
for item in prop_cert:
    if item['String_value_for_toggle']:
        dict_to_be_used = {"PROPERTY_NAME":item['PROPERTY_NAME'],
                            "VALUE":item['String_value_for_toggle']}
        if item['PROPERTY_NAME'] not in properties_considered:
            properties_considered.append(item['PROPERTY_NAME'])
            if dict_to_be_used['VALUE'][0]=='NB':
                dict_to_be_used['VALUE'].append("no break")
                dict_to_be_used['VALUE'].append("nobreak")
            porp_categorical.append(dict_to_be_used)
porp_categorical.append({'PROPERTY_NAME': 'Charpy',
  'VALUE': ['NB', 'no break','nobreak']})
porp_categorical.append({'PROPERTY_NAME': 'rohs',
  'VALUE': ['complaint','+863 compliant', '']})
porp_categorical.append({'PROPERTY_NAME': 'non halogenated',
  'VALUE': ['']})
property_categorical_value_combinations = []

thickness_names = ["thickness", "thickness", "thick.", "thickn."]
thickness_values = ["6", "3", "3.2", "2", "2.5", "1.7", "1.6", "1.5", "1.1", "0.9", "0.75", "0.8", "0.6", "0.5", 
 "0.4", "0.43", "0.71", "0.7", "0.38", "0.2", "0.25", "0.15", "0.18","0.05", "0.10"]
mm_value = ["mm"," mm"]
random_thickness_value = random.choice(thickness_values) + random.choice(mm_value) +" "+random.choice(thickness_names)
for item in porp_categorical:
    for value in item['VALUE']:
        property_categorical_value_combinations.append(item['PROPERTY_NAME'].lower()+ " " +value.lower())
        property_categorical_value_combinations.append(value.lower() + " " + item['PROPERTY_NAME'].lower())
        
for item in ul_categorical:
    for value in item['Categorical_Values']:
        try:
            property_categorical_value_combinations.append(item['UL_PROPERTY'].lower()+ " " +value.lower())
            property_categorical_value_combinations.append(value.lower() + " " + item['UL_PROPERTY'].lower())
        except:
            property_categorical_value_combinations.append(item['UL_SUB_PROPERTY'].lower()+ " " +value.lower())
            property_categorical_value_combinations.append(value.lower() + " " + item['UL_SUB_PROPERTY'].lower())
            property_categorical_value_combinations.append(item['UL_SUB_PROPERTY'].lower() + f" at {random_thickness_value} " + (value.lower()))

ul_numerical.extend([{'UL_PROPERTY': 'Relative Thermal Index – Mechanical Strength',
  'Min_Value': 50.0,
  'Max_Value': 240.0,
  'Avg_value': 145.0,
  'UNIT_OF_MEAS_SI': '°C'},
{'UL_PROPERTY': 'RTI Str',
  'Min_Value': 50.0,
  'Max_Value': 240.0,
  'Avg_value': 145.0,
  'UNIT_OF_MEAS_SI': '°C'},
{'UL_PROPERTY': 'Relative Thermal Index – Mechanical Impact',
  'Min_Value': 50.0,
  'Max_Value': 220.0,
  'Avg_value': 135.0,
  'UNIT_OF_MEAS_SI': '°C'},
  {'UL_PROPERTY': 'RTI Imp',
  'Min_Value': 50.0,
  'Max_Value': 220.0,
  'Avg_value': 135.0,
  'UNIT_OF_MEAS_SI': '°C'},
  {'UL_PROPERTY': 'Relative Thermal Index – Electrical',
  'Min_Value': 50.0,
  'Max_Value': 240.0,
  'Avg_value': 145.0,
  'UNIT_OF_MEAS_SI': '°C'},
  {'UL_PROPERTY': 'RTI Elec',
  'Min_Value': 50.0,
  'Max_Value': 240.0,
  'Avg_value': 145.0,
  'UNIT_OF_MEAS_SI': '°C'},
  {'UL_PROPERTY': 'Glow Wire Ignition Temperature',
  'Min_Value': 625.0,
  'Max_Value': 985.0,
  'Avg_value': 805.0,
  'UNIT_OF_MEAS_SI': '°C'},
  {'UL_PROPERTY': 'gwit',
  'Min_Value': 625.0,
  'Max_Value': 985.0,
  'Avg_value': 805.0,
  'UNIT_OF_MEAS_SI': '°C'},
  {'UL_PROPERTY': 'Glow Wire Flammability Index',
  'Min_Value': 650.0,
  'Max_Value': 960.0,
  'Avg_value': 805.0,
  'UNIT_OF_MEAS_SI': '°C'},
  {'UL_PROPERTY': 'gwfi',
  'Min_Value': 650.0,
  'Max_Value': 960.0,
  'Avg_value': 805.0,
  'UNIT_OF_MEAS_SI': '°C'},
  {'UL_PROPERTY': 'cti',
  'Min_Value': 3.0,
  'Max_Value': 600.0,
  'Avg_value': 301.5,
  'UNIT_OF_MEAS_SI': 'V'},
  ])

{'name': 'instance-0000000005', 'cluster_name': 'eecb414cd4044754b13fc3adefd89c27', 'cluster_uuid': '3yRqD83mT_2OIPt6w_pxsA', 'version': {'number': '8.11.3', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': '64cf052f3b56b1fd4449f5454cb88aca7e739d9a', 'build_date': '2023-12-08T11:33:53.634979452Z', 'build_snapshot': False, 'lucene_version': '9.8.0', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'}
Minimum Thickness (mm) 0.1 6.0 mm
Relative Thermal Index – Mechanical Strength (RTI Str) (°C) 50.0 240.0 °C
Relative Thermal Index – Mechanical Impact (RTI Imp) (°C) 50.0 220.0 °C
Relative Thermal Index – Electrical (RTI Elec) (°C) 50.0 240.0 °C
Glow Wire Ignition Temperature 625.0 985.0 °C
Glow Wire Flammability Index 650.0 960.0 °C
Flame Rating ['5VA', '5VB', 'V-0', 'V-1', 'V-2', 'HB']
Flammability Classification ['5VA', '5VB', 'V-0', 'V-1', 'V-2', 'HB', 'HB40', 'HB75']
Hot Wire Ignition (HWI) 

D:\Users\DSCRS3\AppData\Local\Temp\3\ipykernel_992\702097120.py:17: DeprecationWarning: Received 'size' via a specific parameter in the presence of a 'body' parameter, which is deprecated and will be removed in a future version. Instead, use only 'body' or only specific paremeters.
  ul_resp = ES.search(index='filter_ul', body=match_all_query, size=100)
D:\Users\DSCRS3\AppData\Local\Temp\3\ipykernel_992\702097120.py:136: DeprecationWarning: Received 'size' via a specific parameter in the presence of a 'body' parameter, which is deprecated and will be removed in a future version. Instead, use only 'body' or only specific paremeters.
  prop_resp = ES.search(index='filter_property', body=match_all_query, size=1000)


### more queries with properties with proper value mapping

In [42]:
#single property  without modifier
user_queries = []
for user_query_template in user_query_templates:
    if "proprty1" in user_query_template and "proprty2" not in user_query_template\
        and "modifier1" not in user_query_template:
        n=100
        #print(user_query_template)
        for i in range(n):
            application_sample = random.sample(applications_list,2)
            property_sample = random.sample(property_categorical_value_combinations,3)
            
            modifiers_sample_from_template = random.sample(modifiers_templates,2)
            modifiers_sample = []
            for modifier in modifiers_sample_from_template:
                value = random.choice([str(random.randint(-100, 10000)), str(random.randint(-100, 10000)), str(round(random.uniform(0, 1),2)), random.choice(["{:.1e}","{:.1E}"]).format(random.randint(1, 15) ** random.randint(-10, 15))])
                min_value = random.randint(1, 10000)
                max_value = random.randint(min_value, 10000)
                unit = random.choice([""," "])+random.choice(units_list)
                modifiers_sample.append(eval(modifier))
            modifiers_sample+=random.sample(modifiers_list,2)
            random.shuffle(modifiers_sample)

            feature_sample = random.sample(features_list,4)

            certification_sample = random.sample(certifications_list,3)
            polymer_type_sample = random.sample(Polymer,3)

            filler_type_sample = random.sample(fillers_list,2)
            
            brand_sample = random.sample(Brands,2)
            
            grade_sample = random.sample(grade_list_extended,2)
            
            if contains_duplicates(property_sample) or contains_duplicates(feature_sample) \
                    or contains_duplicates(polymer_type_sample) \
                    or contains_duplicates(filler_type_sample) \
                    or contains_duplicates(grade_sample):
                    continue
            else:
                variable_values = {'application1': application_sample[0],
                'application2': application_sample[1],

                'proprty1': property_sample[0],
                'proprty2': property_sample[1],
                'proprty3': property_sample[2],

                'modifier1': modifiers_sample[0],
                'modifier2': modifiers_sample[1],
                'modifier3': modifiers_sample[2],

                'brand1': brand_sample[0],
                'brand2': brand_sample[1],

                'polymer_type1':polymer_type_sample[0],
                'polymer_type2':polymer_type_sample[1],
                'polymer_type3':polymer_type_sample[2],

                'feature1': feature_sample[0],
                'feature2': feature_sample[1],
                'feature3': feature_sample[2],
                'feature4': feature_sample[3],

                'filler_type1': filler_type_sample[0],
                'filler_type2': filler_type_sample[1],

                'filler_percentage1': random.choice(["","","","","","","","","","","","","","~","=","= ",">=",">= ",">","> ","lower than ", "less than ","greater than ","higher than "]) + \
                                str(random.choice([random.choice(range(0,100)), round(random.uniform(0,100),0)])) + \
                                random.choice([" %","%"," percentage"," percent","", "",""]),
                'filler_percentage2': random.choice(["","","","","","","","","","","","","","~","=","= ",">=",">= ",">","> ","lower than ", "less than ","greater than ","higher than "]) + \
                                str(random.choice([random.choice(range(0,100)), round(random.uniform(0,100),0)])) + \
                                random.choice([" %","%"," percentage"," percent","", "",""]),

                'application_sentences': random.choice(application_sentencess),
                'item': random.choice(items),
                'competitor_grade' :random.choice(competitor_grade_names_extended),
                
                'grade1': grade_sample[0],
                'grade2': grade_sample[1],

                'certification1': certification_sample[0],
                'certification2': certification_sample[1],
                'certification3': certification_sample[2]}

                user_query = eval(user_query_template)
                user_query = user_query.lower()
                entities = []
                
                # Find all variables in the f-string
                matches = list(re.finditer(r"{(.*?)}", user_query_template))
                # Calculate the new start/end positions
                entities = []
                offset = 0
                for match in matches:
                    variable = match.group(1)
                    name_pattern = re.compile(r"\['(.+?)'\]")
                    name_of_variable = name_pattern.search(variable).group(1)
                    value = eval(variable)
                    start_position = match.start()-2 + offset
                    end_position = start_position + len(str(value))
                    if name_of_variable not in ["item","start_sentence","application_sentences"]:
                        entities.append((start_position, end_position, entity_names[name_of_variable]))
                    # Update the offset for the next match
                    offset += len(str(value)) - (match.end() - match.start())

                if min([entity[0] for entity in entities])!=-1:
                    user_queries.append((user_query, {'entities': entities}))
        #         print((user_query, {'entities': entities}))
        #         print('\n')

In [43]:

from sklearn.model_selection import train_test_split
user_queries_train, user_queries_test = train_test_split( user_queries, test_size=0.25, random_state=42)
len(user_queries_train), len(user_queries_test)

(3831, 1278)

In [44]:
train_data+=user_queries_train
test_data+=user_queries_test
len(train_data), len(test_data)

(655184, 59644)

In [45]:
#single property  without modifier
user_queries = []
for user_query_template in user_query_templates:
    if "proprty1" in user_query_template and "modifier1" in user_query_template\
        and "proprty2" in user_query_template  and "modifier2" in user_query_template:
        n=15
        #print(user_query_template)
        for i in range(n):
            application_sample = random.sample(applications_list,2)
            
            property_modifier_samples = []
            for item in ul_numerical:
                if "UL_PROPERTY" in item.keys():
                    item['UL_PROPERTY'].lower()
                    if random.choice([True,False]):
                        if random.choice([True,False]) and item['UNIT_OF_MEAS_SI']:
                            value = str(random.randint(round(item['Min_Value'],0), round(item['Max_Value'],0))) +random.choice(['',' '])+item['UNIT_OF_MEAS_SI']
                        else:
                            value = str(random.randint(round(item['Min_Value'],0), round(item['Max_Value'],0)))
                    else:
                        if random.choice([True,False]) and item['UNIT_OF_MEAS_SI']:
                            value= str(round(random.uniform(item['Min_Value'], item['Max_Value']),1))+random.choice(['',' '])+item['UNIT_OF_MEAS_SI']
                        else:
                            value= str(round(random.uniform(item['Min_Value'], item['Max_Value']),1))
                    property_modifier_samples.append((item['UL_PROPERTY'].lower(),value))
            
            random.shuffle(property_modifier_samples)
            property_modifier_sample = random.sample(property_modifier_samples,3)

            feature_sample = random.sample(features_list,4)
            certification_sample = random.sample(certifications_list,3)
            polymer_type_sample = random.sample(Polymer,3)

            filler_type_sample = random.sample(fillers_list,2)
            
            brand_sample = random.sample(Brands,2)
            
            grade_sample = random.sample(grade_list_extended,2)
            
            if contains_duplicates(property_sample) or contains_duplicates(feature_sample) \
                    or contains_duplicates(polymer_type_sample) \
                    or contains_duplicates(filler_type_sample) \
                    or contains_duplicates(grade_sample):
                    continue
            else:
                variable_values = {'application1': application_sample[0],
                'application2': application_sample[1],

                'proprty1': property_modifier_sample[0][0],
                'proprty2': property_modifier_sample[1][0],
                'proprty3': property_modifier_sample[2][0],

                'modifier1': property_modifier_sample[0][1],
                'modifier2': property_modifier_sample[1][1],
                'modifier3': property_modifier_sample[2][1],

                'brand1': brand_sample[0],
                'brand2': brand_sample[1],

                'polymer_type1':polymer_type_sample[0],
                'polymer_type2':polymer_type_sample[1],
                'polymer_type3':polymer_type_sample[2],

                'feature1': feature_sample[0],
                'feature2': feature_sample[1],
                'feature3': feature_sample[2],
                'feature4': feature_sample[3],

                'filler_type1': filler_type_sample[0],
                'filler_type2': filler_type_sample[1],

                'filler_percentage1': random.choice(["","","","","","","","","","","","","","~","=","= ",">=",">= ",">","> ","lower than ", "less than ","greater than ","higher than "]) + \
                                str(random.choice([random.choice(range(0,100)), round(random.uniform(0,100),0)])) + \
                                random.choice([" %","%"," percentage"," percent","", "",""]),
                'filler_percentage2': random.choice(["","","","","","","","","","","","","","~","=","= ",">=",">= ",">","> ","lower than ", "less than ","greater than ","higher than "]) + \
                                str(random.choice([random.choice(range(0,100)), round(random.uniform(0,100),0)])) + \
                                random.choice([" %","%"," percentage"," percent","", "",""]),

                'application_sentences': random.choice(application_sentencess),
                'item': random.choice(items),
                'competitor_grade' :random.choice(competitor_grade_names_extended),
                
                'grade1': grade_sample[0],
                'grade2': grade_sample[1],

                'certification1': certification_sample[0],
                'certification2': certification_sample[1],
                'certification3': certification_sample[2]}

                user_query = eval(user_query_template)
                user_query = user_query.lower()
                entities = []
                
                # Find all variables in the f-string
                matches = list(re.finditer(r"{(.*?)}", user_query_template))
                # Calculate the new start/end positions
                entities = []
                offset = 0
                for match in matches:
                    variable = match.group(1)
                    name_pattern = re.compile(r"\['(.+?)'\]")
                    name_of_variable = name_pattern.search(variable).group(1)
                    value = eval(variable)
                    start_position = match.start()-2 + offset
                    end_position = start_position + len(str(value))
                    if name_of_variable not in ["item","start_sentence","application_sentences"]:
                        entities.append((start_position, end_position, entity_names[name_of_variable]))
                    # Update the offset for the next match
                    offset += len(str(value)) - (match.end() - match.start())

                if min([entity[0] for entity in entities])!=-1:
                    user_queries.append((user_query, {'entities': entities}))
        #         print((user_query, {'entities': entities}))
        #         print('\n')

In [46]:
from sklearn.model_selection import train_test_split
user_queries_train, user_queries_test = train_test_split( user_queries, test_size=0.25, random_state=42)
len(user_queries_train), len(user_queries_test)

(4653, 1552)

In [47]:
train_data+=user_queries_train
test_data+=user_queries_test
len(train_data), len(test_data)


(659837, 61196)

In [48]:

with open ('./NER Model 650K - 22-02-24/annotated/user_queries_train.txt', 'w', encoding="utf-8") as file:  
    for line in train_data: 
        #print(line)
        file.write(str(line))
        file.write('\n')

with open ('./NER Model 650K - 22-02-24/annotated/user_queries_test.txt', 'w', encoding="utf-8") as file:  
    for line in test_data: 
        #print(line)
        file.write(str(line))
        file.write('\n')

In [49]:
# pd.DataFrame(random.sample(train_data,10000),columns=["Query","Label"]).to_csv("Sample queries.csv",index=False)

#### ul specific templates with thickness

In [48]:
thickness_names = ["thickness", "minimum thickness", "min thickness", "thick.", "min thick.", "thickn.", "min thickn."]
thickness_values = ["6", "3", "3.2", "2", "2.5", "1.7", "1.6", "1.5", "1.1", "0.9", "0.75", "0.8", "0.6", "0.5", 
 "0.4", "0.43", "0.71", "0.7", "0.38", "0.2", "0.25", "0.15", "0.18","0.05", "0.10"]

all_ul_cat_values = []
for item in ul_categorical:
    all_ul_cat_values.extend(item['Categorical_Values'])
all_ul_cat_values = list(set(all_ul_cat_values))

all_ul_cat_values = list(map(lambda x: x.replace('f2 (Suitable for ultraviolet light, water, or immersion exposure)', 'f2'), all_ul_cat_values))
all_ul_cat_values = list(map(lambda x: x.replace('f1 (Suitable for ultraviolet light, water, and immersion exposure', 'f1'), all_ul_cat_values))
property_categorical_value_combinations.extend(all_ul_cat_values)



random.shuffle(property_categorical_value_combinations)

In [49]:
with open ('Query templates/ul_templates_categorical.txt', 'r') as file:  
    ul_templates_cat_with_thickness=[line.strip() for line in file if line.startswith('f"')]
    ul_templates_cat_with_thickness = list(set(ul_templates_cat_with_thickness))
    

In [50]:
#ul categorical with thickness
user_queries = []
for user_query_template in ul_templates_cat_with_thickness:
        n=150
        #print(user_query_template)
        for i in range(n):
            application_sample = random.sample(applications_list,2)
            property_sample = random.sample(property_list,3)
        
            modifiers_sample_from_template = random.sample(modifiers_templates,2)
            modifiers_sample = []
            for modifier in modifiers_sample_from_template:
                value = random.choice([str(random.randint(-100, 10000)), str(random.randint(-100, 10000)), str(round(random.uniform(0, 1),2)), random.choice(["{:.1e}","{:.1E}"]).format(random.randint(1, 15) ** random.randint(-10, 15))])
                min_value = random.randint(1, 10000)
                max_value = random.randint(min_value, 10000)
                unit = random.choice([""," "])+random.choice(units_list)
                modifiers_sample.append(eval(modifier))
            modifiers_sample+=random.sample(modifiers_list,2)
            random.shuffle(modifiers_sample)

            feature_sample = random.sample(features_list,4)

            certification_sample = random.sample(certifications_list,3)
            polymer_type_sample = random.sample(Polymer,3)

            filler_type_sample = random.sample(fillers_list,2)
            
            brand_sample = random.sample(Brands,2)
            
            grade_sample = random.sample(grade_list_extended,2)
            
            thicknesss_min = round(random.uniform(0,5) ,2)
            thickness_range = str(thicknesss_min) +" to "+ str(round(random.uniform(thicknesss_min,6),2)) + random.choice([" mm","mm"])
            

            
            if contains_duplicates(property_sample) or contains_duplicates(feature_sample) \
                    or contains_duplicates(polymer_type_sample) \
                    or contains_duplicates(filler_type_sample) \
                    or contains_duplicates(grade_sample):
                    continue
            else:
                variable_values = {'application1': application_sample[0],
                'application2': application_sample[1],

                'proprty1': property_sample[0],
                'proprty2': property_sample[1],
                'proprty3': property_sample[2],

                'modifier1': modifiers_sample[0],
                'modifier2': modifiers_sample[1],
                'modifier3': modifiers_sample[2],

                'brand1': brand_sample[0],
                'brand2': brand_sample[1],

                'polymer_type1':polymer_type_sample[0],
                'polymer_type2':polymer_type_sample[1],
                'polymer_type3':polymer_type_sample[2],

                'feature1': feature_sample[0],
                'feature2': feature_sample[1],
                'feature3': feature_sample[2],
                'feature4': feature_sample[3],

                'filler_type1': filler_type_sample[0],
                'filler_type2': filler_type_sample[1],

                'filler_percentage1': random.choice(["","","","","","","","","","","","","","~","=","= ",">=",">= ",">","> ","lower than ", "less than ","greater than ","higher than "]) + \
                                str(random.choice([random.choice(range(0,100)), round(random.uniform(0,100),0)])) + \
                                random.choice([" %","%"," percentage"," percent","", "",""]),
                'filler_percentage2': random.choice(["","","","","","","","","","","","","","~","=","= ",">=",">= ",">","> ","lower than ", "less than ","greater than ","higher than "]) + \
                                str(random.choice([random.choice(range(0,100)), round(random.uniform(0,100),0)])) + \
                                random.choice([" %","%"," percentage"," percent","", "",""]),

                'application_sentences': random.choice(application_sentencess),
                'item': random.choice(items),
                'competitor_grade' :random.choice(competitor_grade_names_extended),
                
                'grade1': grade_sample[0],
                'grade2': grade_sample[1],

                'certification1': certification_sample[0],
                'certification2': certification_sample[1],
                'certification3': certification_sample[2],
                                  
                'ul_property_name_value': random.choice(property_categorical_value_combinations),
                'thickness_property_name': random.choice(thickness_names),
                'thickness_modifier':  random.choice([random.choice(thickness_values)+random.choice([" mm","mm"]),
                            random.choice(thickness_values)+random.choice([" mm","mm"]),
                            random.choice(thickness_values)+random.choice([" mm","mm"]),
                            random.choice(thickness_values)+random.choice([" mm","mm"]),
                            random.choice(thickness_values)+random.choice([" mm","mm"]),
                            thickness_range])
                                   
                                  }

                user_query = eval(user_query_template)
                user_query = user_query.lower()
                entities = []
                
                # Find all variables in the f-string
                matches = list(re.finditer(r"{(.*?)}", user_query_template))
                # Calculate the new start/end positions
                entities = []
                offset = 0
                for match in matches:
                    variable = match.group(1)
                    name_pattern = re.compile(r"\['(.+?)'\]")
                    name_of_variable = name_pattern.search(variable).group(1)
                    value = eval(variable)
                    start_position = match.start()-2 + offset
                    end_position = start_position + len(str(value))
                    if name_of_variable not in ["item","start_sentence","application_sentences"]:
                        entities.append((start_position, end_position, entity_names[name_of_variable]))
                    # Update the offset for the next match
                    offset += len(str(value)) - (match.end() - match.start())

                if min([entity[0] for entity in entities])!=-1:
                    user_queries.append((user_query, {'entities': entities}))
#                 print((user_query, {'entities': entities}))
        #         print('\n')

In [51]:
from sklearn.model_selection import train_test_split
user_queries_train, user_queries_test = train_test_split( user_queries, test_size=0.25, random_state=42)
len(user_queries_train), len(user_queries_test)


(1070, 357)

In [52]:
train_data+=user_queries_train
test_data+=user_queries_test
len(train_data), len(test_data)

(660907, 61553)

In [53]:
   
with open ('Query templates/ul_templates_numeric.txt', 'r') as file:  
    ul_templates_numeric_with_thickness=[line.strip() for line in file if line.startswith('f"')]
    ul_templates_numeric_with_thickness = list(set(ul_templates_numeric_with_thickness))
    

In [54]:
#ul categorical numercal with thickness
user_queries = []
for user_query_template in ul_templates_numeric_with_thickness:
        n=150
        #print(user_query_template)
        for i in range(n):
            application_sample = random.sample(applications_list,2)
            property_sample = random.sample(property_list,3)
        
            modifiers_sample_from_template = random.sample(modifiers_templates,2)
            modifiers_sample = []
            for modifier in modifiers_sample_from_template:
                value = random.choice([str(random.randint(-100, 10000)), str(random.randint(-100, 10000)), str(round(random.uniform(0, 1),2)), random.choice(["{:.1e}","{:.1E}"]).format(random.randint(1, 15) ** random.randint(-10, 15))])
                min_value = random.randint(1, 10000)
                max_value = random.randint(min_value, 10000)
                unit = random.choice([""," "])+random.choice(units_list)
                modifiers_sample.append(eval(modifier))
            modifiers_sample+=random.sample(modifiers_list,2)
            random.shuffle(modifiers_sample)

            feature_sample = random.sample(features_list,4)

            certification_sample = random.sample(certifications_list,3)
            polymer_type_sample = random.sample(Polymer,3)

            filler_type_sample = random.sample(fillers_list,2)
            
            brand_sample = random.sample(Brands,2)
            
            grade_sample = random.sample(grade_list_extended,2)
            
            thicknesss_min = round(random.uniform(0,5) ,2)
            thickness_range = str(thicknesss_min) +" to "+ str(round(random.uniform(thicknesss_min,6),2)) + random.choice([" mm","mm"])
            
            property_modifier_samples = []
            for item in ul_numerical:
                if "UL_PROPERTY" in item.keys():
                    item['UL_PROPERTY'].lower()
                    if random.choice([True,False]):
                        if random.choice([True,False]) and item['UNIT_OF_MEAS_SI']:
                            value = str(random.randint(round(item['Min_Value'],0), round(item['Max_Value'],0))) +random.choice(['',' '])+item['UNIT_OF_MEAS_SI']
                        else:
                            value = str(random.randint(round(item['Min_Value'],0), round(item['Max_Value'],0)))
                    else:
                        if random.choice([True,False]) and item['UNIT_OF_MEAS_SI']:
                            value= str(round(random.uniform(item['Min_Value'], item['Max_Value']),1))+random.choice(['',' '])+item['UNIT_OF_MEAS_SI']
                        else:
                            value= str(round(random.uniform(item['Min_Value'], item['Max_Value']),1))
                    property_modifier_samples.append((item['UL_PROPERTY'].lower(),value))

            random.shuffle(property_modifier_samples)
            ul_prop_modifier = random.choice(property_modifier_samples)
            
            if contains_duplicates(property_sample) or contains_duplicates(feature_sample) \
                    or contains_duplicates(polymer_type_sample) \
                    or contains_duplicates(filler_type_sample) \
                    or contains_duplicates(grade_sample):
                    continue
            else:
                variable_values = {'application1': application_sample[0],
                'application2': application_sample[1],

                'proprty1': property_sample[0],
                'proprty2': property_sample[1],
                'proprty3': property_sample[2],

                'modifier1': modifiers_sample[0],
                'modifier2': modifiers_sample[1],
                'modifier3': modifiers_sample[2],

                'brand1': brand_sample[0],
                'brand2': brand_sample[1],

                'polymer_type1':polymer_type_sample[0],
                'polymer_type2':polymer_type_sample[1],
                'polymer_type3':polymer_type_sample[2],

                'feature1': feature_sample[0],
                'feature2': feature_sample[1],
                'feature3': feature_sample[2],
                'feature4': feature_sample[3],

                'filler_type1': filler_type_sample[0],
                'filler_type2': filler_type_sample[1],

                'filler_percentage1': random.choice(["","","","","","","","","","","","","","~","=","= ",">=",">= ",">","> ","lower than ", "less than ","greater than ","higher than "]) + \
                                str(random.choice([random.choice(range(0,100)), round(random.uniform(0,100),0)])) + \
                                random.choice([" %","%"," percentage"," percent","", "",""]),
                'filler_percentage2': random.choice(["","","","","","","","","","","","","","~","=","= ",">=",">= ",">","> ","lower than ", "less than ","greater than ","higher than "]) + \
                                str(random.choice([random.choice(range(0,100)), round(random.uniform(0,100),0)])) + \
                                random.choice([" %","%"," percentage"," percent","", "",""]),

                'application_sentences': random.choice(application_sentencess),
                'item': random.choice(items),
                'competitor_grade' :random.choice(competitor_grade_names_extended),
                
                'grade1': grade_sample[0],
                'grade2': grade_sample[1],

                'certification1': certification_sample[0],
                'certification2': certification_sample[1],
                'certification3': certification_sample[2],
                                  
                'ul_property_name': ul_prop_modifier[0],
                'ul_property_value':ul_prop_modifier[1],
                'thickness_property_name': random.choice(thickness_names),
                'thickness_modifier':  random.choice([random.choice(thickness_values)+random.choice([" mm","mm"]),
                            random.choice(thickness_values)+random.choice([" mm","mm"]),
                            random.choice(thickness_values)+random.choice([" mm","mm"]),
                            random.choice(thickness_values)+random.choice([" mm","mm"]),
                            random.choice(thickness_values)+random.choice([" mm","mm"]),
                            thickness_range])
                                   
                                  }

                user_query = eval(user_query_template)
                user_query = user_query.lower()
                entities = []
                
                # Find all variables in the f-string
                matches = list(re.finditer(r"{(.*?)}", user_query_template))
                # Calculate the new start/end positions
                entities = []
                offset = 0
                for match in matches:
                    variable = match.group(1)
                    name_pattern = re.compile(r"\['(.+?)'\]")
                    name_of_variable = name_pattern.search(variable).group(1)
                    value = eval(variable)
                    start_position = match.start()-2 + offset
                    end_position = start_position + len(str(value))
                    if name_of_variable not in ["item","start_sentence","application_sentences"]:
                        entities.append((start_position, end_position, entity_names[name_of_variable]))
                    # Update the offset for the next match
                    offset += len(str(value)) - (match.end() - match.start())

                if min([entity[0] for entity in entities])!=-1:
                    user_queries.append((user_query, {'entities': entities}))
#                 print((user_query, {'entities': entities}))
        #         print('\n')

In [55]:
from sklearn.model_selection import train_test_split
user_queries_train, user_queries_test = train_test_split( user_queries, test_size=0.25, random_state=42)
len(user_queries_train), len(user_queries_test)


(1088, 363)

In [56]:
train_data+=user_queries_train
test_data+=user_queries_test
len(train_data), len(test_data)

(661995, 61916)

In [57]:

with open ('./NER Model 650K - 22-02-24/annotated/user_queries_train.txt', 'w', encoding="utf-8") as file:  
    for line in train_data: 
        #print(line)
        file.write(str(line))
        file.write('\n')

with open ('./NER Model 650K - 22-02-24/annotated/user_queries_test.txt', 'w', encoding="utf-8") as file:  
    for line in test_data: 
        #print(line)
        file.write(str(line))
        file.write('\n')